In [1]:
import numpy as np
from pyscf import gto, scf, mp, cc

lno_num = 4
lno_list = [3e-4,1e-4,3e-5,1e-5]
lno_thresh = lno_list[lno_num-1]

####  test H2 monomers ####
a = 2 # bond length in a cluster
d = 100 # distance between each cluster
unit = 'b' # unit of length
na = 2 # size of a cluster (monomer)
nc = 3 # set as integer multiple of monomers
spin = 2 # spin per monomer
elmt = 'O'
unit = 'B'
basis = 'sto6g'
atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms,
            basis=basis,
            verbose=4,
            unit=unit,
            symmetry=0,
            charge=0,
            spin=spin*nc,
            max_memory=40000,
            )

mf = scf.UHF(mol).density_fit()
mf.kernel()

stable = False
while not stable:
    print(f'mean-field stability test')
    if not stable:
        mo_i, _, stable,_ = mf.stability(return_status=True)
        dm = mf.make_rdm1(mo_i,mf.mo_occ)
        mf.kernel(dm0=dm)
    elif stable:
        print(f'UHF Energy: {mf.e_tot}, stability {stable}')
        break

mymp = mp.MP2(mf).set_frozen()
mymp.kernel()

mycc = cc.CCSD(mf).set_frozen()
mycc.kernel()

print(f"HF   : {mf.e_tot}")
print(f"MP2  : {mymp.e_tot}")
print(f"CCSD : {mycc.e_tot}")

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-35-generic', version='#35~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue May 26 19:30:42 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Wed Jun 24 14:43:44 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 6
[INPUT] num. electrons = 48
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 6
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      uni

In [2]:
print(f"HF per M   : {mf.e_tot/nc}")
print(f"MP2 per M  : {mymp.e_tot/nc}")
print(f"CCSD per M : {mycc.e_tot/nc}")

HF per M   : -148.96426925758496
MP2 per M  : -149.0279490250154
CCSD per M : -149.039825862244


In [3]:
import jax
jax.config.update("jax_enable_x64", True)
import opt_einsum as oe

In [ ]:
from afqmc.lno_afqmc import lno_afqmc, tools
from pyscf.data import elements
lo_coeff, frag_lolist, atm_center = tools.iao_localization(mf)

In [7]:
options = {
           'n_prop_steps': 50,
           'n_blocks': 600,
           'n_walkers': 300,
           'max_memory': 2000,
           'mix_precision': False,
           'n_batch': 1,
           'seed': 17,
           'walker_type': 'uhf',
           'trial': 'upt2ccsd',
           }

In [9]:
import numpy as np
from jax import random

from pyscf import lib
from pyscf.lno import lnoccsd
from pyscf.lno import ulnoccsd
from collections.abc import Iterable

# from afqmc import config
from afqmc.lno_afqmc import lno_afqmc
from afqmc.lno_afqmc import prep, tools
from afqmc.lno_afqmc import mod_lnoccsd

from functools import partial
import time, gc, pickle, os

print = partial(print, flush=True)

# mf
# lo_coeff = None
# frag_lolist = None
nfrozen = elements.chemcore(mf.mol)
thresh = 1e-5
qmc_options = options
chol_cut = 1e-5
target_sto_error = 0
run_frag_list = [0] 
atom_group = atm_center
plot_las = False

spin_type = prep.kind(lo_coeff)

if frag_lolist is None:
    if spin_type == "unrestricted":
        raise ValueError("frag_lolist must be provided for unrestricted LNO-AFQMC.")
    print("Fragment list not found. Asign every LO to a fragment.")
    frag_lolist = [[i] for i in range(lo_coeff.shape[1])]

if spin_type == "unrestricted":
    mlno = ulnoccsd.ULNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
    mf = mlno._scf
else:
    mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
mlno.lno_thresh = [thresh*10, thresh]
lno_thresh = mlno.lno_thresh
lno_type = ['1h','1h']
eris = mlno.ao2mo()

nfrag_tot = len(frag_lolist)
if run_frag_list is None:
    run_frag_list = range(nfrag_tot)

frag_lolist = [frag_lolist[i] for i in run_frag_list]
nfrag_run = len(frag_lolist)

lno_pct_occ = [None, None]
lno_norb = [[None,None]] * nfrag_tot

seeds = random.randint(random.PRNGKey(qmc_options["seed"]),
                        shape=(nfrag_tot,), 
                        minval=0, 
                        maxval=100*nfrag_tot
                        )

qmc_options["max_error"] = target_sto_error / np.sqrt(nfrag_tot)
trial_base = qmc_options.get("trial", "")

las_center = [None]*nfrag_run
las_size = [None]*nfrag_run
lno_emp2 = np.zeros(nfrag_run, dtype='float64')
lno_ecc  = np.zeros(nfrag_run, dtype='float64')
lno_eqmc = np.zeros(nfrag_run, dtype='float64')
lno_eqmc_err  = np.zeros(nfrag_run, dtype='float64')
ccsd_time = np.zeros(nfrag_run, dtype='float64')
qmc_time = np.zeros(nfrag_run, dtype='float64')

mol = mf.mol

# Loop over fragment
for ifrag, loidx in enumerate(frag_lolist):
    print("\n")
    width = 80
    msg = f" {spin_type} LNO-FRAGMENT {run_frag_list[ifrag]+1}/({nfrag_run},{nfrag_tot}) "
    print(msg.center(width, '='))
    if atom_group is not None:
        loc_ctr = f"{atom_group[run_frag_list[ifrag]]}"
        print(f"Center Atom {loc_ctr}")
    else:
        loc_ctr = None
    
    print(f"PySCF NumPy Threads = {lib.num_threads()}")

    if spin_type == "unrestricted":
        orbloc = [lo_coeff[0][:,loidx[0]], lo_coeff[1][:,loidx[1]]]
        lno_param = [
            [
                {
                    'thresh': (
                        lno_thresh[i][s] if isinstance(lno_thresh[i], Iterable)
                        else lno_thresh[i]
                    ),
                    'pct_occ': (
                        lno_pct_occ[i][s] if isinstance(lno_pct_occ[i], Iterable)
                        else lno_pct_occ[i]
                    ),
                    'norb': (
                        lno_norb[ifrag][i][s] if isinstance(lno_norb[ifrag][i], Iterable)
                        else lno_norb[ifrag][i]
                    ),
                } for i in [0, 1]
            ] for s in range(2)
        ]
    else:
        orbloc = lo_coeff[:,loidx]
        lno_param = [{
            'thresh': lno_thresh[i],
            'pct_occ': lno_pct_occ[i],
            'norb': lno_norb[ifrag][i]
            } for i in [0,1]]

    # M = <orbloc|canactocc> (M^dagger M)u = eu
    # u|canactocc> => orbtial in/out the space spanned by |orbloc>
    # uocc_loc = <lno_actocc|orbloc>
    lno_coeff, lno_frozen, uocc_loc, _ = mlno.make_las(eris, orbloc, lno_type, lno_param)

    mo_occ = mlno.mo_occ

    if spin_type == "unrestricted":
        if uocc_loc[0].size > 0 and uocc_loc[1].size == 0:
            lno_elec_type = 'alpha'
        elif uocc_loc[0].size == 0 and uocc_loc[1].size > 0:
            lno_elec_type = 'beta'
        else:
            lno_elec_type = 'mixed'
        print(f'LNO-Frgament Spin Type = {lno_elec_type}')

        if loc_ctr is None:
            ao_max_a = prep.ao_comp(mf, orbloc[0])
            ao_max_b = prep.ao_comp(mf, orbloc[1])
            loc_ctr = ao_max_a + ao_max_b
            print(f"LNO Center {loc_ctr}")

        lno_frozen, maskact = ulnoccsd.get_maskact(lno_frozen, [mo_occ[0].size, mo_occ[1].size])
        occidxa = mo_occ[0] > 1e-10
        occidxb = mo_occ[1] > 1e-10
        moidxa, moidxb = maskact
        nactocc_a = int(np.sum(moidxa & occidxa))
        nactvir_a = int(np.sum(moidxa & ~occidxa))
        nactocc_b = int(np.sum(moidxb & occidxb))
        nactvir_b = int(np.sum(moidxb & ~occidxb))
        nactocc = [nactocc_a, nactocc_b]
        nactvir = [nactvir_a, nactvir_b]
        lno_active_a = np.array([i for i in range(mol.nao) if i not in lno_frozen[0]])
        lno_active_b = np.array([i for i in range(mol.nao) if i not in lno_frozen[1]])
        lno_active = [lno_active_a, lno_active_b]
        lno_tot = [len(lno_active_a), len(lno_active_b)]
        # print(f'LAS alpha: {nactocc_a} occupied, {nactvir_a} virtual')
        print(f'LAS occupied orbitals:  {nactocc}')
        print(f'LAS virtual orbitals:   {nactvir}')
        print(f'LAS total size:         {lno_tot}')
        mcc = ulnoccsd.UCCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    else:
        print(f'LNO-Frgament Spin Type = restricted')
        if loc_ctr is None:
            loc_ctr = prep.ao_comp(mf, orbloc)
            print(f"LNO Center {loc_ctr}")

        lno_frozen, maskact = lnoccsd.get_maskact(lno_frozen, mo_occ.size)
        lno_active = np.array([i for i in range(mol.nao) if i not in lno_frozen])
        nactocc, nactvir = prep.las_size(mf, lno_frozen)
        lno_tot = len(lno_active)
        print(f'LAS occupied orbitals:  {nactocc}')
        print(f'LAS virtual orbitals:   {nactvir}')
        print(f'LAS total size:         {lno_tot}')
        mcc = lnoccsd.CCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    
    if plot_las:
        tools.plot_density(mol, orbloc, lno_coeff, lno_active, spin_type, idx = run_frag_list[ifrag]+1)

    mcc._s1e = mlno._s1e
    mcc._h1e = mlno._h1e
    mcc._vhf = mlno._vhf
    if mlno.kwargs_imp is not None:
        mcc = mcc.set(**mlno.kwargs_imp)
    time0 = time.perf_counter()
    (eorb_mp2, eorb_cc), t1, t2 =\
        mod_lnoccsd.lnoccsd_kernel(mcc, lno_coeff, uocc_loc, mo_occ, maskact)
    time1 = time.perf_counter()
    lnocc_time = time1 - time0

    print(f'LNO-MP2 Orbital Energy:  {eorb_mp2:.8f}')
    print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')
    print(f"LNO-CCSD time:           {lnocc_time:.2f} s")

    las_center[ifrag] = loc_ctr
    las_size[ifrag] = lno_tot
    lno_emp2[ifrag] = eorb_mp2
    lno_ecc[ifrag] = eorb_cc
    ccsd_time[ifrag] = lnocc_time

    # project onto center lo space
    # <lno_actocc|orbloc> <orbloc|lno_actocc>
    if spin_type == "unrestricted":
        prjlo = [uocc_loc[0] @ uocc_loc[0].T.conj(),
                    uocc_loc[1] @ uocc_loc[1].T.conj()]
        qmc_options["trial"] = trial_base
        if 'ad' not in trial_base:
            if lno_elec_type == 'alpha':
                qmc_options["trial"] += '_alpha'
            elif lno_elec_type == 'beta':
                qmc_options["trial"] += '_beta'
    else:
        prjlo = uocc_loc @ uocc_loc.T.conj()

    qmc_options["seed"] = seeds[ifrag]
    prep.prep_afqmc_integral(
        mf,
        lno_coeff,
        t1,
        t2,
        lno_frozen,
        prjlo,
        qmc_options,
        chol_cut=chol_cut
        )
    
    lno_afqmc.run_lnoafqmc(qmc_options)
    outfile = f'fragment.out{run_frag_list[ifrag]+1}'
    os.system(f'mv afqmc.out {outfile}')
    with open(outfile, "r") as f:
        for line in f:
            if "Blocked AFQMC/pt2CCSD Orbital Energy" in line:
                eorb_afqmc = float(line.split()[-3])
                eorb_afqmc_err = float(line.split()[-1])
            if "total run time" in line:
                lnoqmc_time = float(line.split()[-1])
    lno_eqmc[ifrag] = eorb_afqmc
    lno_eqmc_err[ifrag] = eorb_afqmc_err
    qmc_time[ifrag] = lnoqmc_time

    header = f' Fragment{run_frag_list[ifrag]+1} Results '
    width = 80  # pick a consistent total width
    with open(outfile, 'a') as f:
        f.write('\n')
        f.write(f'{header:=^{width}}\n')
        f.write("\t LNO Center " + loc_ctr + "\n")
        f.write('-' * width + '\n')
        f.write(f'\t LNO-Active Space electrons: {nactocc} | orbitals: {nactocc+nactvir} \n')
        f.write(f'\t LNO-MP2 Orbital Energy:   {eorb_mp2:.8f} \n')
        f.write(f'\t LNO-CCSD Orbital Energy:  {eorb_cc:.8f} \n')
        f.write(f'\t LNO-AFQMC Orbital Energy: {eorb_afqmc:.6f} +/- {eorb_afqmc_err:.6f} \n')
        f.write(f'\t LNO-CCSD Time:  {lnocc_time:.2f} \n')
        f.write(f'\t LNO-AFQMC Time: {lnoqmc_time:.2f} \n')
        f.write('=' * width + '\n')
    jax.clear_caches()
    gc.collect()

las_size = np.array(las_size, dtype=np.int32)
las_max = las_size.max()
# convert to list of string for print
las_size = list(map(lambda row: f"{row}", las_size))
e_mp2 = np.sum(lno_emp2)
e_ccsd = np.sum(lno_ecc)
e_afqmc = np.sum(lno_eqmc)
e_afqmc_err = np.sqrt(np.sum(lno_eqmc_err**2))
tot_ccsd_time = np.sum(ccsd_time)
tot_qmc_time = np.sum(qmc_time)

with open(f'lno_result.out', 'w') as f:
    width = 100
    f.write('=' * width + '\n')
    f.write(f'{"LNO-AFQMC Results":^{width}}\n')
    f.write('=' * width + '\n')

    f.write(f'{"Frag":>4s}  {"LAS Center":>14s}  {"LAS_SIZE":>8s}  '
            f'{"E(MP2)":>10s}  {"E(CCSD)":>10s}  '
            f'{"E(AFQMC)":>10s}  {"Error":>8s}  '
            f'{"t(CCSD)":>8s}  {"t(AFQMC)":>8s}\n')
    f.write('-' * width + '\n')
    
    for n, i in enumerate(run_frag_list):
        f.write(f"{i+1:4d}  {las_center[n]:>14s}  {las_size[n]:8s}  "
                f"{lno_emp2[n]:10.8f}  {lno_ecc[n]:10.8f}  "
                f"{lno_eqmc[n]:10.6f}  {lno_eqmc_err[n]:8.6f}  "
                f"{ccsd_time[n]:8.2f}  {qmc_time[n]:8.2f}\n")
    
    f.write('-' * width + '\n')
    f.write(f'{"Summarize Fragments":^{width}}\n')
    f.write('-' * width + '\n')
    lno_thresh_str = "[" + ", ".join(f"{x:.2e}" for x in lno_thresh) + "]"
    f.write(f'{"LNO-Thresh":<20} {"Max LAS":>8} '
            f'{"E[MP2]":>12} {"E[CCSD]":>12} '
            f'{"E[AFQMC]":>10} {"Err[AFQMC]":>10} '
            f'{"CCSD-Time":>10} {"AFQMC-Time":>10}\n')

    f.write(f'{lno_thresh_str:<20} {las_max:>8} '
            f'{e_mp2:>12.8f} {e_ccsd:>12.8f} '
            f'{e_afqmc:>10.6f} {e_afqmc_err:>10.6f} '
            f'{tot_ccsd_time:>10.2f} {tot_qmc_time:>10.2f}\n')
    
    f.write('=' * width + '\n\n')



====================== unrestricted LNO-FRAGMENT 1/(1,1) =======================
Center Atom O
PySCF NumPy Threads = 16
LNO-Frgament Spin Type = mixed
LAS occupied orbitals:  [7, 5]
LAS virtual orbitals:   [1, 3]
LAS total size:         [8, 8]
LNO-MP2 Orbital Energy:  -0.03183988
LNO-CCSD Orbital Energy: -0.03777830
LNO-CCSD time:           0.06 s
Calculating Effective Active Space One-electron Integrals
Building JK matrix
build JK time: 0.009234 s
build ecore time: 0.003309 s
build h1eff time: 0.014299 s
Generating Cholesky Integrals
Constracting cLAS that span both Alpha and Beta active space
Naive cLAS Shape:  (30, 16)
Orthonormalize cLAS shape: (30, 16)
Smallest cLAS SVD Singular values: 3.5227824607389853e-16
cLAS projection torr: 1e-08
Minimum size of cLAS to span both Alpha and Beta LAS: 10
cLAS projection loss: (2.67e-12, 2.74e-12)
True Common LAS Shape:  (30, 10)
Composing AO ERIs from DF basis
Raw CDERI in cLAS shape: (516, 55)
Cholesky cutoff is: 1e-05
Compress CDERI into 

In [21]:
ham_data, prop, trial, wave_data, sampler, options = (prep.prep_afqmc_run())
wave_data["rdm1"] = trial.get_rdm1(wave_data)
ham_data = trial._build_measurement_intermediates(ham_data, wave_data)
ham_data = prop._build_propagation_intermediates(ham_data, trial, wave_data)
prop_data = prop.init_prop_data(trial, wave_data, ham_data, init_walkers = None)



QMC System
Number of electrons:        (7, 5)
Number of orbitals:         (8, 8)
Number of Cholesky Vectors: 40


Maximum memory per walker:            6.67 MB
Maximum number of Cholesky per chunk: 3413
Number of Cholesky chunks:            1
Number of Cholesky per chunk:         40
Number of padding Cholesky:           0

QMC Parameters
n_prop_steps    -              50
n_blocks        -             600
n_walkers       -             300
max_memory      -            2000
mix_precision   -           False
n_batch         -               1
seed            -              42
walker_type     -             uhf
trial           -        upt2ccsd
max_error       -               0
dt              -           0.005
eql_time        -              20
nchol_chunk     -              40
n_exp_terms     -               6


In [22]:
ham_data["h1_mod"]

KeyError: 'h1_mod'

In [15]:
t1a, t1b = mycc.t1
t2aa, t2ab, t2bb = mycc.t2
e0t1 = mycc.energy((t1a, t1b),(t2aa*0, t2ab*0, t2bb*0))
print(e0t1, e0t1/nc/2)
print(ham_data["e0t1orb"])

1.5469211552415777e-05 2.5782019254026293e-06
2.5781711017744374e-06


In [18]:
walkers_up, walkers_dn = prop_data["walkers"]
walker_up = walkers_up[0]; walker_dn = walkers_dn[0]
e0bar_f = trial._calc_eorb_bar(walker_up, walker_dn, ham_data, wave_data)
print(e0bar_f)

(2.5781711017744374e-06+0j)


In [32]:
from afqmc import slater_tools
from jax import jit, jvp
from jax import numpy as jnp

@jit
def u_olp_exp1(x: float, h1_mod: tuple, bra:tuple, ket:tuple):
    '''
    unrestricted <bra|exp(x*h1_mod)|ket>/<bra|ket>
    '''
    ket_up_1x = ket[0] + x * h1_mod[0].dot(ket[0])
    ket_dn_1x = ket[1] + x * h1_mod[1].dot(ket[1])
    ket1x = (ket_up_1x, ket_dn_1x)
    o0 = slater_tools.u_overlap(bra, ket)
    o1x = slater_tools.u_overlap(bra, ket1x)

    return o1x/o0

@jit
def u_olp_exp2(x: float, chol_i: tuple, bra: tuple, ket: tuple) -> complex:
    '''
    <bra|exp(x*h2_mod)|ket>/<bra|ket>
    '''
    ket_up_2x = (
        ket[0] + x * chol_i[0].dot(ket[0]) 
        + x**2 / 2.0 * chol_i[0].dot(chol_i[0].dot(ket[0]))
        )
    ket_dn_2x = (
        ket[1] + x * chol_i[1].dot(ket[1])
        + x**2 / 2.0 * chol_i[1].dot(chol_i[1].dot(ket[1]))
        )
    
    ket2x = (ket_up_2x, ket_dn_2x)
    o0 = slater_tools.u_overlap(bra, ket)
    o2x = slater_tools.u_overlap(bra, ket2x)
    
    return o2x/o0

@jit
def _d2_u_olp_exp2_i(chol_i, bra, ket):
    x = 0.0
    f = lambda a: u_olp_exp2(a,chol_i, bra, ket)
    _, d2f = jax.jvp(lambda x: jax.jvp(f, [x], [1.0])[1], [x], [1.0])
    return d2f

@jit
def _d2_u_olp_exp2(chol, bra, ket):
    d2_exp2_batch = jax.vmap(_d2_u_olp_exp2_i, in_axes=(0,None,None))
    d2_exp2s = d2_exp2_batch(chol, bra, ket)
    h2 = jnp.sum(d2_exp2s)/2
    return h2

@partial(jit, static_argnums=0)
def _calc_energy_ad(self, walker_up, walker_dn, ham_data, wave_data):
    '''
    energy = e0 + e1 +e2
    e0 = h0
    e1 = <trial|h1|walker>/<trial|walker>
    e2 = <trial|h2|walker>/<trial|walker>
    '''

    norba, norbb = self.norb
    h1_mod = ham_data['h1_mod']
    chola, cholb = ham_data["chol"]
    chola = chola.reshape(-1, norba, norba)
    cholb = cholb.reshape(-1, norbb, norbb)
    chol = (chola, cholb)

    bra = wave_data["mo_coeff"]
    ket = (walker_up, walker_dn)

    # one body
    x = 0.0
    f1 = lambda a: u_olp_exp1(a, h1_mod, bra, ket)
    olp, e1 = jvp(f1, [x], [1.0])

    # two body
    e2 = _d2_u_olp_exp2(chol, bra, ket)
    e = ham_data["h0"] + e1 + e2

    return e

In [28]:
trial.norb

(8, 8)

In [26]:
h1 = ham_data["h1"]
norba, norbb = h1[0].shape[0], h1[1].shape[0]
chola, cholb = ham_data["chol"]
chola = chola.reshape(-1, norba, norba)
cholb = cholb.reshape(-1, norbb, norbb)
v0_a = 0.5 * oe.contract("gik,gjk->ij", chola, chola, backend='jax')
v0_b = 0.5 * oe.contract("gik,gjk->ij", cholb, cholb, backend='jax')
h1mod_a = np.array(h1[0]) - v0_a
h1mod_b = np.array(h1[1]) - v0_b
ham_data["h1_mod"] = (h1mod_a, h1mod_b)

In [33]:
_calc_energy_ad(trial, walker_up, walker_dn, ham_data, wave_data)

Array(-446.89279763+0.j, dtype=complex128)

In [34]:
print(mf.e_tot)

-446.89280777275485
